In [4]:
from pathlib import Path
import pandas as pd

data_path = Path("LA_listings.csv")
if not data_path.exists():
    data_path = Path("project") / "LA_listings.csv"

if not data_path.exists():
    raise FileNotFoundError("Cannot find LA_listings.csv")

raw = pd.read_csv(
    data_path,
    dtype={
        "id": "string",
        "host_id": "string",
        "host_profile_id": "string"
    }
)

print("File:", data_path.resolve())
print("Rows and columns:", raw.shape)

display(raw.head())
display(raw.dtypes.rename("data_type").to_frame())

missing = raw.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].rename("missing_count").to_frame())

File: /Users/chen/dso576/project/LA_listings.csv
Rows and columns: (43932, 19)


,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,2708,"Run Runyon, Beaut Furn Mirror Mini-Suit w/ Fir...",3008,1462506333679450606,⁨Chas.⁩,City of Los Angeles,Hollywood,34.09625,-118.34605,Private room,67.0,30.0,47,2026-05-26,0.32,2.0,301,3,NaN
1,2732,Zen Life at the Beach,3041,1462506335238008418,Yogapriestess,Other Cities,Santa Monica,34.00440,-118.48095,Private room,214.0,1.0,24,2022-08-21,0.13,2.0,350,0,228269
2,6033,Poolside Serenity Studio,11619,1462506670443088558,Sarah,City of Los Angeles,Woodland Hills,34.16887,-118.64478,Entire home/apt,100.0,30.0,19,2020-03-22,0.09,7.0,270,0,NaN
3,6931,"RUN Runyon, Beau Furn Rms Terrace Hollyw Hill ...",3008,1462506333679450606,⁨Chas.⁩,City of Los Angeles,Hollywood,34.09626,-118.34372,Private room,95.0,30.0,39,2025-12-15,0.19,2.0,259,2,NaN
4,7874,HD Wi-Fi E Bellflower Cerritos Lakewood LB LA,21700,1462506951665098500,Henry,Other Cities,Bellflower,33.87687,-118.11444,Private room,113.0,1.0,26,2026-04-02,0.29,4.0,103,9,NaN


,data_type
id,string[python]
name,object
host_id,string[python]
host_profile_id,string[python]
host_name,object
neighbourhood_group,object
neighbourhood,object
latitude,float64
longitude,float64
room_type,object


,missing_count
license,30740
reviews_per_month,11311
last_review,11311
price,5570
host_profile_id,395
host_name,384
host_id,181
calculated_host_listings_count,181
minimum_nights,11


Project vision: Airbnb’s Los Angeles marketplace team wants to identify neighborhood and room-type segments where existing listings may benefit from pricing or listing guidance.  
Business question: Which Los Angeles neighborhood and room-type segments have low recent review activity alongside high annual availability, and what is the median nightly price in those segments?  
Main measures: number_of_reviews_ltm, availability_365, and median price.

In [5]:
print("Total rows:", len(raw))
print("Missing listing IDs:", raw["id"].isna().sum())
print("Exact duplicate rows:", raw.duplicated().sum())

id_counts = raw["id"].value_counts()
repeated_ids = id_counts[id_counts > 1]

print("Listing IDs that appear more than once:", len(repeated_ids))
print("Extra rows from repeated IDs:", int((repeated_ids - 1).sum()))

if len(repeated_ids) > 0:
    display(
        raw[raw["id"].isin(repeated_ids.index)]
        .sort_values("id")
        .head(10)
    )

Total rows: 43932
Missing listing IDs: 0
Exact duplicate rows: 0
Listing IDs that appear more than once: 0
Extra rows from repeated IDs: 0


Row definition and duplicate check
Each row represents one Los Angeles Airbnb listing in this listings file. id is the listing identifier. Among 43,932 rows, there were 0 missing listing IDs, 0 exact duplicate rows, and 0 repeated listing IDs. I kept all rows; no rows were removed for duplication.

In [6]:
# Check whether nonmissing last_review values can be parsed as dates
parsed_dates = pd.to_datetime(raw["last_review"], errors="coerce")
invalid_dates = raw["last_review"].notna() & parsed_dates.isna()

print("Invalid nonmissing review dates:", int(invalid_dates.sum()))
print("Earliest parsed review date:", parsed_dates.min())
print("Latest parsed review date:", parsed_dates.max())

# Check whether these float columns contain fractional values
for column in ["minimum_nights", "calculated_host_listings_count"]:
    values = raw[column]
    fractional = values.notna() & values.mod(1).ne(0)
    print(f"{column} non-integer values:", int(fractional.sum()))

# Check potentially suspicious numeric values
checks = {
    "price <= 0": raw["price"].le(0).sum(),
    "minimum_nights <= 0": raw["minimum_nights"].le(0).sum(),
    "availability outside 0-365": (
        raw["availability_365"].notna()
        & ~raw["availability_365"].between(0, 365)
    ).sum(),
    "negative number_of_reviews": raw["number_of_reviews"].lt(0).sum(),
    "number_of_reviews_ltm > number_of_reviews": (
        raw["number_of_reviews_ltm"] > raw["number_of_reviews"]
    ).sum(),
}

for check, count in checks.items():
    print(f"{check}: {int(count)}")

Invalid nonmissing review dates: 0
Earliest parsed review date: 2011-05-21 00:00:00
Latest parsed review date: 2026-06-22 00:00:00
minimum_nights non-integer values: 0
calculated_host_listings_count non-integer values: 0
price <= 0: 0
minimum_nights <= 0: 0
availability outside 0-365: 0
negative number_of_reviews: 0
number_of_reviews_ltm > number_of_reviews: 0


In [7]:
for column in ["last_review", "reviews_per_month"]:
    print(f"\n{column}: missingness vs. zero reviews")

    display(
        pd.crosstab(
            raw["number_of_reviews"].eq(0),
            raw[column].isna(),
            rownames=["number_of_reviews == 0"],
            colnames=[f"{column} is missing"],
            margins=True
        )
    )


last_review: missingness vs. zero reviews


last_review is missing,False,True,All
number_of_reviews == 0,,,
False,32621,0,32621
True,0,11311,11311
All,32621,11311,43932



reviews_per_month: missingness vs. zero reviews


reviews_per_month is missing,False,True,All
number_of_reviews == 0,,,
False,32621,0,32621
True,0,11311,11311
All,32621,11311,43932


Missing-review fields
Rule: Keep missing values in last_review and reviews_per_month when number_of_reviews is 0. Do not impute them in the raw data.
Reason: These listings have no review history, so a last-review date and historical monthly review rate are not available. The primary analysis uses number_of_reviews_ltm, which has no missing values.
Check: In the full dataset, all 11,311 listings with zero lifetime reviews have both fields missing. All 32,621 listings with at least one review have both fields present.

In [8]:
missing_counts = raw.isna().sum()
display(
    missing_counts[missing_counts > 0]
    .rename("missing_count")
    .to_frame()
)

price_by_segment = (
    raw.assign(price_missing=raw["price"].isna())
    .groupby(
        ["neighbourhood_group", "room_type"],
        dropna=False
    )
    .agg(
        listings=("id", "size"),
        missing_prices=("price_missing", "sum"),
        missing_price_share=("price_missing", "mean"),
        valid_prices=("price", "count")
    )
    .sort_values("missing_price_share", ascending=False)
)

display(price_by_segment)

,missing_count
host_id,181
host_profile_id,395
host_name,384
price,5570
minimum_nights,11
last_review,11311
reviews_per_month,11311
calculated_host_listings_count,181
license,30740


listings  missing_prices  \
neighbourhood_group  room_type                                   
Unincorporated Areas Hotel room             18               4   
                     Private room          985             190   
Other Cities         Private room         4531             660   
City of Los Angeles  Entire home/apt     16133            2273   
                     Private room         5255             676   
Other Cities         Entire home/apt     13254            1448   
Unincorporated Areas Entire home/apt      3067             296   
Other Cities         Shared room            73               4   
City of Los Angeles  Shared room           203               8   
Other Cities         Hotel room            220               7   
City of Los Angeles  Hotel room            177               4   
Unincorporated Areas Shared room            16               0   

                                      missing_price_share  valid_prices  
neighbourhood_group  room_type                                           
Unincorporated Areas Hotel room                  0.222222            14  
                     Private room                0.192893           795  
Other Cities         Private room                0.145663          3871  
City of Los Angeles  Entire home/apt             0.140891         13860  
                     Private room                0.128639          4579  
Other Cities         Entire home/apt             0.109250         11806  
Unincorporated Areas Entire home/apt             0.096511          2771  
Other Cities         Shared room                 0.054795            69  
City of Los Angeles  Shared room                 0.039409           195  
Other Cities         Hotel room                  0.031818           213  
City of Los Angeles  Hotel room                  0.022599           173  
Unincorporated Areas Shared room                 0.000000            16

Other missing values
Rule: Keep rows with missing price; calculate each segment’s median price from its nonmissing prices and report the valid price count. Keep missing license, host information, and minimum_nights values; do not impute them or drop those listings.
Reason: Missing prices occur at different rates across neighborhood and room-type segments, so dropping rows could change the group comparisons. The other fields are not required for the selected primary measures.
Check: The full dataset has 5,570 missing prices (12.68%); missing-price shares vary by segment. All displayed neighborhood-group and room-type segments have observed prices. license has 30,740 missing values; host_id 181; host_profile_id 395; host_name 384; calculated_host_listings_count 181; and minimum_nights 11.

In [9]:
text_columns = [
    "name",
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "room_type",
    "license"
]

for column in text_columns:
    values = raw[column].astype("string")
    has_extra_spaces = values.notna() & values.ne(values.str.strip())

    print(f"{column}: values with leading/trailing spaces = "
          f"{int(has_extra_spaces.sum())}")

    if has_extra_spaces.any():
        display(
            raw.loc[has_extra_spaces, column]
            .drop_duplicates()
            .head(10)
            .to_frame()
        )

print("\nRoom types:")
display(raw["room_type"].value_counts(dropna=False).to_frame("count"))

print("\nNeighborhood groups:")
display(
    raw["neighbourhood_group"]
    .value_counts(dropna=False)
    .to_frame("count")
)

name: values with leading/trailing spaces = 16


,name
27404,C3. Balcony Room | Queen Bed · Shared Bath
27407,C4. Premier Queen Ensuite Room | Private Bath
27408,C6. Corner Ensuite Room | Queen Bed
27409,D3. Balcony Room | Queen Bed · Shared Bath
30505,B2. Primary Suite | Queen Bed · Private Bath
31214,C1 MidCity Exclusive Master Suite w/Private Bath
36921,B6. Corner Ensuite Room | Queen Bed
36993,B. MidCity Grand Retreat: 6 Rooms & 5 Baths
36995,A3. Balcony Room | Queen Bed · Shared Bath
37267,A4. Premier Queen Ensuite Room | Private Bath


host_name: values with leading/trailing spaces = 0
neighbourhood_group: values with leading/trailing spaces = 0
neighbourhood: values with leading/trailing spaces = 0
room_type: values with leading/trailing spaces = 0
license: values with leading/trailing spaces = 0

Room types:


,count
room_type,
Entire home/apt,32454
Private room,10771
Hotel room,415
Shared room,292



Neighborhood groups:


,count
neighbourhood_group,
City of Los Angeles,21768
Other Cities,18078
Unincorporated Areas,4086


In [10]:
name_values = raw["name"].astype("string")
name_has_extra_spaces = (
    name_values.notna()
    & name_values.ne(name_values.str.strip())
)

for row_index in raw.index[name_has_extra_spaces]:
    original = raw.at[row_index, "name"]
    cleaned = original.strip()
    print(f"row {row_index}: {original!r} -> {cleaned!r}")

row 27404: 'C3. Balcony Room | Queen Bed · Shared Bath\u2028' -> 'C3. Balcony Room | Queen Bed · Shared Bath'
row 27407: 'C4. Premier Queen Ensuite Room | Private Bath\u2028' -> 'C4. Premier Queen Ensuite Room | Private Bath'
row 27408: 'C6. Corner Ensuite Room | Queen Bed\u2028' -> 'C6. Corner Ensuite Room | Queen Bed'
row 27409: 'D3. Balcony Room | Queen Bed · Shared Bath\u2028' -> 'D3. Balcony Room | Queen Bed · Shared Bath'
row 30505: 'B2. Primary Suite | Queen Bed · Private Bath\u2028' -> 'B2. Primary Suite | Queen Bed · Private Bath'
row 31214: 'C1 MidCity Exclusive Master Suite w/Private Bath\xa0' -> 'C1 MidCity Exclusive Master Suite w/Private Bath'
row 36921: 'B6. Corner Ensuite Room | Queen Bed\u2028' -> 'B6. Corner Ensuite Room | Queen Bed'
row 36993: 'B. MidCity Grand Retreat: 6 Rooms & 5 Baths\xa0' -> 'B. MidCity Grand Retreat: 6 Rooms & 5 Baths'
row 36995: 'A3. Balcony Room | Queen Bed · Shared Bath\u2028' -> 'A3. Balcony Room | Queen Bed · Shared Bath'
row 37267: 'A4. Pr

Listing names
Rule: Remove leading and trailing whitespace from name with .str.strip(). Preserve internal spaces, capitalization, punctuation, and wording.
Reason: Sixteen names contain trailing Unicode whitespace characters; these are formatting artifacts rather than part of the listing names.
Check: I compared the original and trimmed values with repr(). Only boundary whitespace was removed. No other inspected text columns had leading or trailing whitespace.

In [11]:
clean = raw.copy()
clean["name"] = clean["name"].str.strip()

changed_name = raw["name"].notna() & raw["name"].ne(clean["name"])

print("Original rows:", len(raw))
print("Clean rows:", len(clean))
print("Names changed:", int(changed_name.sum()))

display(
    pd.DataFrame({
        "original": raw.loc[changed_name, "name"],
        "cleaned": clean.loc[changed_name, "name"]
    }).head(10)
)

Original rows: 43932
Clean rows: 43932
Names changed: 16


,original,cleaned
27404,C3. Balcony Room | Queen Bed · Shared Bath,C3. Balcony Room | Queen Bed · Shared Bath
27407,C4. Premier Queen Ensuite Room | Private Bath,C4. Premier Queen Ensuite Room | Private Bath
27408,C6. Corner Ensuite Room | Queen Bed,C6. Corner Ensuite Room | Queen Bed
27409,D3. Balcony Room | Queen Bed · Shared Bath,D3. Balcony Room | Queen Bed · Shared Bath
30505,B2. Primary Suite | Queen Bed · Private Bath,B2. Primary Suite | Queen Bed · Private Bath
31214,C1 MidCity Exclusive Master Suite w/Private Bath,C1 MidCity Exclusive Master Suite w/Private Bath
36921,B6. Corner Ensuite Room | Queen Bed,B6. Corner Ensuite Room | Queen Bed
36993,B. MidCity Grand Retreat: 6 Rooms & 5 Baths,B. MidCity Grand Retreat: 6 Rooms & 5 Baths
36995,A3. Balcony Room | Queen Bed · Shared Bath,A3. Balcony Room | Queen Bed · Shared Bath
37267,A4. Premier Queen Ensuite Room | Private Bath,A4. Premier Queen Ensuite Room | Private Bath


Data types
Rule: Keep id, host_id, and host_profile_id as strings. Convert last_review to a date type. Convert minimum_nights and calculated_host_listings_count to nullable integers (Int64).
Reason: IDs are labels, dates should be usable as dates, and the two count fields contain only whole numbers. Nullable integers preserve their missing values.
Check: All three ID columns loaded as strings. All 32,621 nonmissing last_review values parsed successfully, with 0 failures. The two count fields have 0 non-integer values; preserve the 11 missing minimum_nights values and 181 missing calculated_host_listings_count values.

In [12]:
# Check date conversion failures before converting
parsed_dates = pd.to_datetime(raw["last_review"], errors="coerce")
date_failures = raw["last_review"].notna() & parsed_dates.isna()

# Apply the documented type conversions to the clean copy
clean["last_review"] = pd.to_datetime(
    clean["last_review"],
    errors="coerce"
)
clean["minimum_nights"] = clean["minimum_nights"].astype("Int64")
clean["calculated_host_listings_count"] = (
    clean["calculated_host_listings_count"].astype("Int64")
)

print("Rows before:", len(raw))
print("Rows after:", len(clean))
print("Date conversion failures:", int(date_failures.sum()))

columns_to_check = [
    "last_review",
    "minimum_nights",
    "calculated_host_listings_count"
]

display(clean[columns_to_check].dtypes.rename("data_type").to_frame())

missing_check = pd.DataFrame({
    "before": raw[columns_to_check].isna().sum(),
    "after": clean[columns_to_check].isna().sum()
})
display(missing_check)

Rows before: 43932
Rows after: 43932
Date conversion failures: 0


,data_type
last_review,datetime64[ns]
minimum_nights,Int64
calculated_host_listings_count,Int64


,before,after
last_review,11311,11311
minimum_nights,11,11
calculated_host_listings_count,181,181


In [13]:
print("Raw shape:", raw.shape)
print("Clean shape:", clean.shape)
print("Rows removed:", len(raw) - len(clean))
print("Exact duplicate rows in clean:", clean.duplicated().sum())
print("Repeated listing IDs in clean:", clean["id"].duplicated().sum())

clean_names = clean["name"].astype("string")
remaining_name_spaces = (
    clean_names.notna()
    & clean_names.ne(clean_names.str.strip())
)
print("Names still with leading/trailing whitespace:",
      int(remaining_name_spaces.sum()))

affected_columns = raw.columns[raw.isna().any()]
missing_comparison = pd.DataFrame({
    "before": raw[affected_columns].isna().sum(),
    "after": clean[affected_columns].isna().sum()
})
display(missing_comparison)

metrics = ["number_of_reviews_ltm", "availability_365", "price"]
metric_counts = pd.DataFrame({
    "valid_records": clean[metrics].count(),
    "missing_records": clean[metrics].isna().sum()
})
display(metric_counts)

Raw shape: (43932, 19)
Clean shape: (43932, 19)
Rows removed: 0
Exact duplicate rows in clean: 0
Repeated listing IDs in clean: 0
Names still with leading/trailing whitespace: 0


,before,after
host_id,181,181
host_profile_id,395,395
host_name,384,384
price,5570,5570
minimum_nights,11,11
last_review,11311,11311
reviews_per_month,11311,11311
calculated_host_listings_count,181,181
license,30740,30740


,valid_records,missing_records
number_of_reviews_ltm,43932,0
availability_365,43932,0
price,38362,5570


### Final cleaning audit

The cleaned dataset contains 43,932 rows and 19 columns, matching the raw dataset. No rows were removed. The cleaned data has no exact duplicate rows or repeated listing IDs, and no listing names retain leading or trailing whitespace.

Missing values were preserved. The two main activity measures, `number_of_reviews_ltm` and `availability_365`, each have 43,932 valid records. `price` has 38,362 valid records and 5,570 missing records; price summaries should report the number of listings with a nonmissing price.

In [14]:
# Build a reproducible sample of up to 50 cleaned records,
# including examples of important cleaning cases.
selected_indices = []

cases = {
    "name was trimmed": raw["name"].astype("string").ne(clean["name"].astype("string")),
    "price is missing": clean["price"].isna(),
    "minimum_nights is missing": clean["minimum_nights"].isna(),
    "no lifetime reviews": clean["number_of_reviews"].eq(0),
    "license is missing": clean["license"].isna(),
    "host_id is missing": clean["host_id"].isna(),
}

# Include one example for each case, when available.
for case, mask in cases.items():
    matches = clean.index[mask]
    for index in matches:
        if index not in selected_indices:
            selected_indices.append(index)
            break

# Include an example of each room type and neighborhood group.
for column in ["room_type", "neighbourhood_group"]:
    for value in clean[column].dropna().unique():
        matches = clean.index[clean[column].eq(value)]
        for index in matches:
            if index not in selected_indices:
                selected_indices.append(index)
                break

# Fill the sample to 50 rows using a fixed random seed.
remaining_indices = clean.index.difference(selected_indices)
needed = min(50 - len(selected_indices), len(remaining_indices))

if needed > 0:
    extra_indices = (
        clean.loc[remaining_indices]
        .sample(n=needed, random_state=576)
        .index
        .tolist()
    )
    selected_indices.extend(extra_indices)

cleaned_sample = clean.loc[selected_indices].copy()

sample_path = data_path.parent / "cleaned_sample.csv"
cleaned_sample.to_csv(sample_path, index=False)

print("Sample rows:", len(cleaned_sample))
print("Saved to:", sample_path)
display(cleaned_sample.head())

Sample rows: 50
Saved to: cleaned_sample.csv


,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
27404,1270001704182989864,C3. Balcony Room | Queen Bed · Shared Bath,657676434,1470652207152838639,Kyusan,City of Los Angeles,West Adams,34.033874,-118.369052,Private room,143.0,1,54,2026-05-25,2.73,30,256,28,NaN
6,9140,City Place Lofts,28350,1462507093368580991,Wendell,Other Cities,Long Beach,33.772020,-118.188520,Private room,NaN,31,389,2019-11-25,2.40,1,0,0,NaN
861,2436587,Polynesian Room,12358058,1462885925242515828,Buddy,City of Los Angeles,Mar Vista,33.995500,-118.444280,Private room,72.0,<NA>,212,2026-03-04,1.44,6,0,1,NaN
46,584724,The Charlie WeHo Luxurious 2BDR/2BATH Apartment,2881219,1462611424198830262,Dylan,City of Los Angeles,Beverly Grove,34.085070,-118.369700,Entire home/apt,1043.0,30,0,NaT,NaN,1,363,0,NaN
0,2708,"Run Runyon, Beaut Furn Mirror Mini-Suit w/ Fir...",3008,1462506333679450606,⁨Chas.⁩,City of Los Angeles,Hollywood,34.096250,-118.346050,Private room,67.0,30,47,2026-05-26,0.32,2,301,3,NaN


In [15]:
def show_value(value):
    if pd.isna(value):
        return "<missing>"
    return repr(value)


checks = []

# 1. Leading and trailing Unicode whitespace in a name should be removed.
idx = 27404
original = raw.loc[idx, "name"]
expected = str(original).strip()
actual = clean.loc[idx, "name"]
checks.append({
    "row": idx,
    "field": "name",
    "original": show_value(original),
    "expected": show_value(expected),
    "actual": show_value(actual),
    "passed": actual == expected,
})

# 2. A missing price should remain missing.
idx = 6
original = raw.loc[idx, "price"]
actual = clean.loc[idx, "price"]
checks.append({
    "row": idx,
    "field": "price",
    "original": show_value(original),
    "expected": "<missing> (preserved)",
    "actual": f"{show_value(actual)} (preserved)",
    "passed": pd.isna(original) and pd.isna(actual),
})

# 3. A missing minimum_nights value should remain missing;
#    the column should use nullable Int64.
idx = 861
original = raw.loc[idx, "minimum_nights"]
actual = clean.loc[idx, "minimum_nights"]
checks.append({
    "row": idx,
    "field": "minimum_nights",
    "original": show_value(original),
    "expected": "<missing> (nullable Int64)",
    "actual": f"{show_value(actual)} ({clean['minimum_nights'].dtype})",
    "passed": (
        pd.isna(original)
        and pd.isna(actual)
        and str(clean["minimum_nights"].dtype) == "Int64"
    ),
})

# 4. A missing last_review value should remain missing after date conversion.
idx = 46
original = raw.loc[idx, "last_review"]
expected = pd.to_datetime(original, errors="coerce")
actual = clean.loc[idx, "last_review"]
checks.append({
    "row": idx,
    "field": "last_review",
    "original": show_value(original),
    "expected": show_value(expected),
    "actual": show_value(actual),
    "passed": pd.isna(expected) and pd.isna(actual),
})

# 5. A listing ID should keep its value and string type.
idx = 0
original = raw.loc[idx, "id"]
actual = clean.loc[idx, "id"]
checks.append({
    "row": idx,
    "field": "id",
    "original": show_value(original),
    "expected": f"{show_value(original)} (string)",
    "actual": f"{show_value(actual)} ({clean['id'].dtype})",
    "passed": (
        actual == original
        and str(clean["id"].dtype).startswith("string")
    ),
})

record_checks = pd.DataFrame(checks)
display(record_checks)
print("All five checks passed:", bool(record_checks["passed"].all()))

,row,field,original,expected,actual,passed
0,27404,name,'C3. Balcony Room | Queen Bed · Shared Bath\u2...,'C3. Balcony Room | Queen Bed · Shared Bath','C3. Balcony Room | Queen Bed · Shared Bath',True
1,6,price,<missing>,<missing> (preserved),<missing> (preserved),True
2,861,minimum_nights,<missing>,<missing> (nullable Int64),<missing> (Int64),True
3,46,last_review,<missing>,<missing>,<missing>,True
4,0,id,'2708','2708' (string),'2708' (string),True


All five checks passed: True


### Five record checks

I compared five records before and after cleaning. The checks confirmed that leading and trailing whitespace was removed from a listing name, missing values in `price`, `minimum_nights`, and `last_review` were preserved, and a listing ID retained its original value and string type. All five checks passed.

In [17]:
from pathlib import Path
import pandas as pd

# Find the original CSV in the current folder or its project subfolder.
current_folder = Path.cwd()
possible_paths = [
    current_folder / "LA_listings.csv",
    current_folder / "project" / "LA_listings.csv",
]
data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        f"Could not find LA_listings.csv in {current_folder} "
        "or its project subfolder."
    )

# Read the original data. The source file is not changed.
raw = pd.read_csv(
    data_path,
    dtype={
        "id": "string",
        "host_id": "string",
        "host_profile_id": "string",
    },
)

# Apply the cleaning rules to a copy.
clean = raw.copy()
clean["name"] = clean["name"].astype("string").str.strip()
clean["last_review"] = pd.to_datetime(
    clean["last_review"], errors="coerce"
)
clean["minimum_nights"] = clean["minimum_nights"].astype("Int64")
clean["calculated_host_listings_count"] = (
    clean["calculated_host_listings_count"].astype("Int64")
)

# Verify that valid dates were not lost during conversion.
date_failures = raw["last_review"].notna() & clean["last_review"].isna()

# Verify row count, duplicate IDs, and name cleanup.
name_has_outer_whitespace = (
    clean["name"].notna()
    & clean["name"].ne(clean["name"].str.strip())
)

assert len(raw) == len(clean), "The row count changed."
assert clean["id"].duplicated().sum() == 0, "Repeated listing IDs found."
assert int(date_failures.sum()) == 0, "Invalid date conversions found."
assert int(name_has_outer_whitespace.sum()) == 0, "Untrimmed names found."

# Pick records that show important cleaning cases.
selected_indices = []
cases = [
    raw["name"].astype("string").ne(clean["name"]).fillna(False),
    clean["price"].isna(),
    clean["minimum_nights"].isna(),
    clean["number_of_reviews"].eq(0),
    clean["license"].isna(),
    clean["host_id"].isna(),
]

for mask in cases:
    for index in clean.index[mask]:
        if index not in selected_indices:
            selected_indices.append(index)
            break

# Include each room type and neighborhood group when available.
for column in ["room_type", "neighbourhood_group"]:
    for value in clean[column].dropna().unique():
        for index in clean.index[clean[column].eq(value)]:
            if index not in selected_indices:
                selected_indices.append(index)
                break

# Fill the sample to 50 rows reproducibly.
remaining_indices = clean.index.difference(selected_indices)
needed = min(50 - len(selected_indices), len(remaining_indices))

if needed > 0:
    selected_indices.extend(
        clean.loc[remaining_indices]
        .sample(n=needed, random_state=576)
        .index.tolist()
    )

cleaned_sample = clean.loc[selected_indices].copy()
sample_path = data_path.parent / "cleaned_sample.csv"
cleaned_sample.to_csv(sample_path, index=False)

print("Raw shape:", raw.shape)
print("Clean shape:", clean.shape)
print("Date conversion failures:", int(date_failures.sum()))
print("Sample rows:", len(cleaned_sample))
print("Sample saved to:", sample_path)

Raw shape: (43932, 19)
Clean shape: (43932, 19)
Date conversion failures: 0
Sample rows: 50
Sample saved to: /Users/chen/dso576/project/cleaned_sample.csv
